# Inferencia

Generación de las predicciones de `SalePrice` para los datos crudos de `test.csv`, usando el modelo ganador del torneo (`models/modelo_ganador.joblib`), que incluye el pipeline completo de feature engineering y fue reentrenado con el 100% de `train.csv`.

Al final se crea el **pipeline de inferencia**: un único objeto de sklearn que recibe datos crudos y devuelve `SalePrice` en dólares, sin volver a entrenar el modelo.

## Librerías

In [ ]:
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from sklearn.compose import TransformedTargetRegressor
from sklearn.frozen import FrozenEstimator

## Configuración

Rutas de las fuentes y de las salidas. Las features, el target, el identificador y la ruta del modelo ganador se leen de `pipeline_config.yaml`.

In [ ]:
# Rutas: modificar estas variables si cambia la ubicación de las fuentes o de las salidas
PROJ_DIR = Path(os.getenv("PROJ_DIR", ".."))
CONFIG_FILE = Path(os.getenv("PIPELINE_CONFIG", PROJ_DIR / "pipeline_config.yaml"))
RAW_DATA_DIR = Path(os.getenv("RAW_DATA_DIR", PROJ_DIR / "data" / "raw"))
PROCESSED_DATA_DIR = Path(os.getenv("PROCESSED_DATA_DIR", PROJ_DIR / "data" / "processed"))
MODELS_DIR = Path(os.getenv("MODELS_DIR", PROJ_DIR / "models"))

TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
SAMPLE_SUBMISSION_FILE = "sample_submission.csv"
PREDICCIONES_FILE = "predicciones_test.csv"
PIPELINE_INFERENCIA_FILE = "pipeline_inferencia.joblib"

with open(CONFIG_FILE, encoding="utf-8") as archivo:
    config = yaml.safe_load(archivo)

MODEL_FEATURES = config["MODEL_FEATURES"]
TARGET = config["TARGET"]
ID_COL = config["ID_COLUMN"]

## Carga de los datos crudos de test

Se carga `test.csv` sin ninguna transformación, con `Id` como índice, y se verifica que traiga todas las `MODEL_FEATURES`. Los vacíos se dejan tal cual: los imputa el pipeline.

In [ ]:
test = pd.read_csv(RAW_DATA_DIR / TEST_FILE, index_col=ID_COL)

faltantes = [col for col in MODEL_FEATURES if col not in test.columns]
if faltantes:
    raise ValueError(f"Faltan columnas en {TEST_FILE}: {faltantes}")

X_test = test[MODEL_FEATURES]

print(f"X_test: {X_test.shape}")
print(f"Vacíos en los datos crudos: {X_test.isna().sum().sum()}")

## Carga del modelo ganador

El archivo contiene el pipeline completo ya ajustado: los 7 pasos de feature engineering y el modelo ganador.

In [ ]:
modelo_ganador = joblib.load(PROJ_DIR / config["BEST_MODEL"]["ruta"])

print(f"Modelo: {config['BEST_MODEL']['nombre']}")
print(f"Entrenado con: {config['BEST_MODEL']['entrenado_con']}")
print(f"Pasos del feature engineering: {[nombre for nombre, _ in modelo_ganador.named_steps['fe'].steps]}")

## Predicciones

El pipeline aplica el feature engineering a los datos crudos y predice en escala log; las predicciones se regresan a dólares con `np.expm1`.

In [ ]:
pred_log = modelo_ganador.predict(X_test)
predicciones = pd.DataFrame({TARGET: np.expm1(pred_log)}, index=X_test.index)

print(f"Predicciones: {len(predicciones)}  |  Vacíos: {predicciones[TARGET].isna().sum()}")
predicciones[TARGET].describe().round(0)

## Guardar las predicciones

Se guardan en `data/processed/` con las columnas `Id` y `SalePrice` (en dólares), y se verifica que tengan el mismo formato y las mismas casas que `sample_submission.csv`.

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
ruta_predicciones = PROCESSED_DATA_DIR / PREDICCIONES_FILE
predicciones.to_csv(ruta_predicciones)

muestra = pd.read_csv(RAW_DATA_DIR / SAMPLE_SUBMISSION_FILE)
guardado = pd.read_csv(ruta_predicciones)

print(f"Mismas columnas que {SAMPLE_SUBMISSION_FILE}: {guardado.columns.tolist() == muestra.columns.tolist()}")
print(f"Mismas casas que {SAMPLE_SUBMISSION_FILE}: {guardado[ID_COL].equals(muestra[ID_COL])}")
print(f"Predicciones guardadas en: {ruta_predicciones.resolve()}")
guardado.head()

## Pipeline de inferencia

Se integra todo el flujo de inferencia en un solo objeto de sklearn:

- `FrozenEstimator` congela el modelo ganador: su `fit` no hace nada, así que el feature engineering y el modelo **no se vuelven a entrenar**.
- `TransformedTargetRegressor` agrega la transformación del target: `np.log1p` al ajustar y `np.expm1` al predecir, de modo que `predict` devuelve `SalePrice` directamente en dólares.

`TransformedTargetRegressor` necesita llamar a `fit` para quedar listo; se le pasa `train.csv` solo para registrar la transformación del target, y el modelo congelado lo ignora. Se verifica que el resultado sea idéntico a las predicciones anteriores.

In [ ]:
train = pd.read_csv(RAW_DATA_DIR / TRAIN_FILE, index_col=ID_COL)

pipeline_inferencia = TransformedTargetRegressor(
    regressor=FrozenEstimator(modelo_ganador),
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)
pipeline_inferencia.fit(train[MODEL_FEATURES], train[TARGET])

predicciones_pipeline = pipeline_inferencia.predict(X_test)

print(f"Idéntico a las predicciones anteriores: "
      f"{np.allclose(predicciones_pipeline, predicciones[TARGET])}")
pipeline_inferencia

## Guardar el pipeline de inferencia

Se guarda en `models/` con joblib y su ruta se agrega a `pipeline_config.yaml`. Para usarlo con datos nuevos:

```python
pipeline_inferencia = joblib.load("models/pipeline_inferencia.joblib")
precios = pipeline_inferencia.predict(datos_crudos[MODEL_FEATURES])  # SalePrice en dólares
```

Al final se compara, en la misma gráfica, la densidad de `SalePrice` real en train con la de `SalePrice` predicho por el pipeline para test. Si el modelo generaliza bien, ambas distribuciones deben parecerse.

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
ruta_pipeline_inferencia = MODELS_DIR / PIPELINE_INFERENCIA_FILE
joblib.dump(pipeline_inferencia, ruta_pipeline_inferencia)


def ruta_relativa(ruta):
    """Ruta relativa a la raíz del proyecto, para guardarla en la configuración."""
    return Path(os.path.relpath(ruta, PROJ_DIR)).as_posix()


config["INFERENCE"] = {
    "pipeline": ruta_relativa(ruta_pipeline_inferencia),
    "salida": f"{TARGET} en dólares",
    "predicciones_test": ruta_relativa(ruta_predicciones),
}

with open(CONFIG_FILE, "w", encoding="utf-8") as archivo:
    yaml.safe_dump(config, archivo, sort_keys=False, allow_unicode=True)

# Verificación: el pipeline cargado desde disco reproduce las predicciones
recargado = joblib.load(ruta_pipeline_inferencia)
print(f"El pipeline cargado reproduce las predicciones: "
      f"{np.allclose(recargado.predict(X_test), predicciones[TARGET])}")
print(f"Pipeline de inferencia guardado en: {ruta_pipeline_inferencia.resolve()}")
print(f"Configuración guardada en: {CONFIG_FILE.resolve()}")

# Comparación de la densidad de SalePrice real (train) contra el predicho por el pipeline (test)
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(x=train[TARGET], fill=True, alpha=0.4, label=f"{TARGET} real (train)", ax=ax)
sns.kdeplot(x=predicciones_pipeline, fill=True, alpha=0.4, label=f"{TARGET} predicho (test)", ax=ax)
ax.set_title(f"Densidad de {TARGET}: real en train vs predicho por el pipeline")
ax.set_xlabel(f"{TARGET} (dólares)")
ax.set_ylabel("Densidad")
ax.legend()
plt.show()